In [1]:
import pandas as pd
from io import StringIO
from pathlib import Path

In [2]:
name = ["FD001", "FD002", "FD003", "FD004"]

for nm in name:
    input_file = "train_%s.txt" % nm
    output_file = "X_Y_%s.csv" % nm

    # Read the input file
    text = Path(input_file).read_text(encoding="utf-8")
    text = text.replace("&#x20;", " ")

    df = pd.read_csv(StringIO(text), sep=r"\s+", header=None)

    # Expected columns: id, cycle, 3 operating conditions, and 21 observations
    if df.shape[1] != 26:
        raise ValueError(f"Detected {df.shape[1]} columns; expected 26 columns.")

    observation_columns = [f"obs_{i}" for i in range(1, 22)]
    condition_columns = [f"condition_{i}" for i in range(1, 4)]

    df.columns = ["id", "cycle"] + condition_columns + observation_columns

    # Calculate RUL independently for each machine
    df["RUL"] = df.groupby("id")["cycle"].transform("max") - df["cycle"]

    # Keep id, obs_1 through obs_21, and RUL
    result = df[["id"] + observation_columns + ["RUL"]]

    result.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"Conversion completed: {output_file}")

Conversion completed: X_Y_FD001.csv
Conversion completed: X_Y_FD002.csv
Conversion completed: X_Y_FD003.csv
Conversion completed: X_Y_FD004.csv
